In [1]:
import os

# Set working directory to project root always
# Works regardless of where the notebook is saved
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
os.chdir(project_root)

print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\DELL\OneDrive\Documents\SRM\Churn_Analysis


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib, os

df = pd.read_csv('data/ecommerce/ECommerce.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing:\n", df.isnull().sum()[df.isnull().sum()>0])

Shape: (5630, 20)
Columns: ['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
Missing:
 Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64


In [3]:
# Drop ID column if it exists
drop_candidates = ['CustomerID', 'customerID', 'Customer ID']
for col in drop_candidates:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)
        print(f"Dropped: {col}")

print("Shape after drop:", df.shape)

Dropped: CustomerID
Shape after drop: (5630, 19)


In [4]:
print("Missing values before:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("Total:", df.isnull().sum().sum())

# Fill numerical nulls with median
for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val:.2f}")

# Fill categorical nulls with mode
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} with mode: {mode_val}")

# Force fill any remaining nulls
df.fillna(df.median(numeric_only=True), inplace=True)

print("\nMissing after fix:", df.isnull().sum().sum())

Missing values before:
Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64
Total: 1856
Filled Tenure with median: 9.00
Filled WarehouseToHome with median: 14.00
Filled HourSpendOnApp with median: 3.00
Filled OrderAmountHikeFromlastYear with median: 15.00
Filled CouponUsed with median: 1.00
Filled OrderCount with median: 2.00
Filled DaySinceLastOrder with median: 3.00

Missing after fix: 0


C:\Users\DELL\AppData\Local\Temp\ipykernel_22460\1896183566.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(median_val, inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_22460\1896183566.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an 

In [5]:
from sklearn.preprocessing import LabelEncoder
import joblib

# Find categorical columns
cat_cols = df.select_dtypes('object').columns.tolist()

print("Categorical columns to encode:")
print(cat_cols)

# Store encoders
encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    encoders[col] = le

    print(
        f"Encoded: {col} | "
        f"Classes: {list(le.classes_)}"
    )

# Save all encoders
joblib.dump(
    encoders,
    'outputs/ecommerce_encoders.pkl'
)

print("\nEncoders saved successfully.")

print(
    "\nObject columns remaining:",
    df.select_dtypes('object').columns.tolist()
)

print("Shape:", df.shape)

Categorical columns to encode:
['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']
Encoded: PreferredLoginDevice | Classes: ['Computer', 'Mobile Phone', 'Phone']
Encoded: PreferredPaymentMode | Classes: ['CC', 'COD', 'Cash on Delivery', 'Credit Card', 'Debit Card', 'E wallet', 'UPI']
Encoded: Gender | Classes: ['Female', 'Male']
Encoded: PreferedOrderCat | Classes: ['Fashion', 'Grocery', 'Laptop & Accessory', 'Mobile', 'Mobile Phone', 'Others']
Encoded: MaritalStatus | Classes: ['Divorced', 'Married', 'Single']

Encoders saved successfully.

Object columns remaining: []
Shape: (5630, 19)


C:\Users\DELL\AppData\Local\Temp\ipykernel_22460\4181405762.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes('object').columns.tolist()


In [6]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print("Features shape:", X.shape)
print("Target shape  :", y.shape)
print("\nChurn distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean()*100:.1f}%")

Features shape: (5630, 18)
Target shape  : (5630,)

Churn distribution:
Churn
0    4682
1     948
Name: count, dtype: int64

Churn rate: 16.8%


In [7]:
num_cols = X.select_dtypes(include='number').columns.tolist()
print("Scaling columns:", num_cols)

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("\nAfter scaling — mean should be ~0:")
print(X[num_cols].mean().round(3))

Scaling columns: ['Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

After scaling — mean should be ~0:
Tenure                         0.0
PreferredLoginDevice          -0.0
CityTier                       0.0
WarehouseToHome                0.0
PreferredPaymentMode           0.0
Gender                        -0.0
HourSpendOnApp                 0.0
NumberOfDeviceRegistered      -0.0
PreferedOrderCat              -0.0
SatisfactionScore             -0.0
MaritalStatus                 -0.0
NumberOfAddress                0.0
Complain                      -0.0
OrderAmountHikeFromlastYear   -0.0
CouponUsed                    -0.0
OrderCount                    -0.0
DaySinceLastOrder             -0.0
CashbackAmount     

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("\nChurn rate train:", round(y_train.mean(), 3))
print("Churn rate test :", round(y_test.mean(), 3))

X_train: (4504, 18)
X_test : (1126, 18)

Churn rate train: 0.168
Churn rate test : 0.169


In [9]:
print("Nulls in X_train:", X_train.isnull().sum().sum())
print("Nulls in y_train:", y_train.isnull().sum())
print("\nColumns with nulls:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])

Nulls in X_train: 0
Nulls in y_train: 0

Columns with nulls:
Series([], dtype: int64)


In [10]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE :", pd.Series(y_train_sm).value_counts().to_dict())
print("New shape   :", X_train_sm.shape)

Before SMOTE: {0: 3746, 1: 758}
After SMOTE : {0: 3746, 1: 3746}
New shape   : (7492, 18)


In [11]:
os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)

np.save('outputs/X_train_ecommerce.npy', X_train_sm)
np.save('outputs/X_test_ecommerce.npy',  X_test.values)
np.save('outputs/y_train_ecommerce.npy', y_train_sm)
np.save('outputs/y_test_ecommerce.npy',  y_test.values)

joblib.dump(scaler, 'outputs/scaler_ecommerce.pkl')

pd.Series(X_train.columns).to_csv(
    'outputs/ecommerce_feature_names.csv', index=False
)

print("All files saved:")
print(os.listdir('outputs/'))

All files saved:
['ecommerce_encoders.pkl', 'ecommerce_feature_names.csv', 'lr_model_ecommerce.pkl', 'lr_model_telecom.pkl', 'plots', 'results', 'rf_model_ecommerce.pkl', 'rf_model_telecom.pkl', 'scaler_ecommerce.pkl', 'scaler_telecom.pkl', 'telecom_feature_names.csv', 'xgb_model_ecommerce.pkl', 'xgb_model_telecom.pkl', 'X_test_ecommerce.npy', 'X_test_telecom.npy', 'X_train_ecommerce.npy', 'X_train_telecom.npy', 'y_test_ecommerce.npy', 'y_test_telecom.npy', 'y_train_ecommerce.npy', 'y_train_telecom.npy']
